# Whisper Small Darija ASR Benchmark (Kaggle T4 GPU)

Compare resource-friendly fine-tuned **Whisper Small** models (244M params) against the resource-heavy **large-v3 with Darija LoRA** adapter (809M params + 50M LoRA) to see if they offer a viable low-resource alternative.

## Models Benchmarked

1. **`ychafiqui/whisper-small-darija`** (Fine-tuned Whisper Small, 244M params)
2. **`Anass-Srk/fine-tuned-whisper-small-darija`** (Fine-tuned Whisper Small, 244M params)
3. **`anaszil/whisper-large-v3-turbo-darija`** (Large-V3-Turbo LoRA, ~859M params - Baseline Champion)

## Setup Instructions
- **Settings** → **Accelerator** → **GPU T4 x2** (or any GPU accelerator)
- **Internet** → **On** (required to download model weights from Hugging Face)
- Click **Run All**

## 1. Install dependencies

In [ ]:
!pip -q install "transformers>=4.47" "torch>=2.0" "accelerate" "librosa" "soundfile" "peft" "jiwer" "tabulate"

## 2. Clone the transcription repository

In [ ]:
import os, sys
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/haca-transcription")
if not REPO_ROOT.exists():
    !git clone --depth 1 https://github.com/MarTCM/haca-transcription.git "{REPO_ROOT}"
else:
    print("[repo] already cloned")

sys.path.insert(0, str(REPO_ROOT / "src"))
print(f"[repo] {REPO_ROOT}")

## 3. Configure audio file

In [ ]:
import os, torch, time, json, gc
import librosa
import soundfile as sf
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/haca-transcription")

# --- CONFIGURE THESE ---
SAMPLE_PATH = str(REPO_ROOT / "samples" / "mobachara_ma3akom_trimmed.mp3")
SAMPLE_URL  = None          # override: download from URL
OUT_DIR     = Path("/kaggle/working/compare_small")
# -----------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)
WAV_PATH = OUT_DIR / "sample_16khz.wav"

if os.path.exists(SAMPLE_PATH):
    src = SAMPLE_PATH
elif SAMPLE_URL:
    src = str(OUT_DIR / "sample_src")
    !wget -q "{SAMPLE_URL}" -O "{src}"
else:
    raise RuntimeError(
        "SAMPLE_PATH not found and no SAMPLE_URL set. Did the git clone fail?"
    )

audio, sr = librosa.load(src, sr=16000, mono=True)
sf.write(str(WAV_PATH), audio, 16000)
duration = len(audio) / 16000
print(f"Audio: {WAV_PATH}  ({duration:.1f}s, {sr} Hz, mono)")

## 4. Benchmark Helper Function

In [ ]:
import sys

def transcribe_and_time(
    model_id: str,
    audio_path: str,
    is_lora: bool = False,
    lora_base: str = "openai/whisper-large-v3-turbo",
) -> dict:
    """Load model_id, transcribe, measure duration, and clean up GPU memory."""
    device = 0 if torch.cuda.is_available() else "cpu"
    dtype  = torch.float16 if torch.cuda.is_available() else torch.float32
    result = {"model": model_id, "load_s": 0, "transcribe_s": 0, "text": ""}

    # ---- Load ----
    t0 = time.time()
    base = None
    model = None
    try:
        if is_lora:
            from peft import PeftModel
            from transformers import (
                WhisperForConditionalGeneration, WhisperProcessor, pipeline,
            )
            base = WhisperForConditionalGeneration.from_pretrained(
                lora_base, torch_dtype=dtype, low_cpu_mem_usage=True,
            )
            model = PeftModel.from_pretrained(base, model_id)
            processor = WhisperProcessor.from_pretrained(
                lora_base, language="Arabic", task="transcribe",
            )
            pipe = pipeline(
                "automatic-speech-recognition",
                model=model,
                tokenizer=processor.tokenizer,
                feature_extractor=processor.feature_extractor,
                chunk_length_s=30,
                device=device,
            )
        else:
            from transformers import pipeline as pl
            pipe = pl(
                "automatic-speech-recognition",
                model=model_id,
                device=device,
                torch_dtype=dtype,
            )
    except Exception as e:
        result["error"] = str(e)
        return result
    result["load_s"] = round(time.time() - t0, 1)

    # ---- Transcribe ----
    t0 = time.time()
    try:
        # For fine-tuned whisper-small models, we explicitly guide ASR to Arabic/transcribe task
        gen_kwargs = {"language": "arabic", "task": "transcribe"} if not is_lora else {}
        out = pipe(audio_path, return_timestamps=True, generate_kwargs=gen_kwargs)
        result["text"] = out["text"].strip()
        result["chunks"] = out.get("chunks", [])
    except Exception as e:
        result["error"] = str(e)
        result["text"] = "[TRANSCRIPTION FAILED]"
    result["transcribe_s"] = round(time.time() - t0, 1)

    # ---- Cleanup ----
    del pipe
    if model is not None:
        del model
    if base is not None:
        del base
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

def pretty_print(result: dict):
    tag = result["model"].split("/")[-1]
    err = result.get("error")
    print(f"\n{'='*60}", file=sys.stderr)
    print(f"  {tag}", file=sys.stderr)
    print(f"  Load: {result['load_s']}s  |  Transcribe: {result['transcribe_s']}s", file=sys.stderr)
    if err:
        print(f"  ERROR: {err}", file=sys.stderr)
    print(f"{'='*60}", file=sys.stderr)
    print(result["text"][:500], "\n..." if len(result["text"]) > 500 else "")

## 5. Run benchmarks

### Model 1: `ychafiqui/whisper-small-darija` (244M params)

In [ ]:
result_1 = transcribe_and_time("ychafiqui/whisper-small-darija", str(WAV_PATH))
pretty_print(result_1)

### Model 2: `Anass-Srk/fine-tuned-whisper-small-darija` (244M params)

In [ ]:
result_2 = transcribe_and_time("Anass-Srk/fine-tuned-whisper-small-darija", str(WAV_PATH))
pretty_print(result_2)

### Baseline Model: `anaszil/whisper-large-v3-turbo-darija` (LoRA, 809M + 50M params)

In [ ]:
result_baseline = transcribe_and_time(
    "anaszil/whisper-large-v3-turbo-darija", str(WAV_PATH),
    is_lora=True,
    lora_base="openai/whisper-large-v3-turbo",
)
pretty_print(result_baseline)

## 6. Comparison Table

In [ ]:
from tabulate import tabulate

rows = []
for r in [result_1, result_2, result_baseline]:
    tag = r["model"].split("/")[-1]
    rows.append([
        tag,
        r.get("load_s", "N/A"),
        r.get("transcribe_s", "N/A"),
        "✔" if r.get("text") else r.get("error", "✘"),
        r["text"][:200] + ("..." if len(r["text"]) > 200 else ""),
    ])

print(tabulate(
    rows,
    headers=["Model (Size)", "Load (s)", "Transcribe (s)", "OK?", "First 200 chars"],
    tablefmt="grid",
))
print(f"\nAudio duration: {duration:.1f}s")

## 7. Compare Word Error Rate (WER) if ground truth is available

In [ ]:
REF_PATH = None  # Replace with path to a ground-truth transcript .txt file if available

if REF_PATH and os.path.exists(REF_PATH):
    from jiwer import wer
    ref_text = Path(REF_PATH).read_text().strip()
    print("WER comparison (lower is better):")
    print(f"  {'Model':<50} {'WER':>8}")
    print(f"  {'-'*50} {'-'*8}")
    for r in [result_1, result_2, result_baseline]:
        hyp = r.get("text", "")
        if hyp:
            w = round(wer(ref_text, hyp), 2)
        else:
            w = "N/A"
        print(f"  {r['model']:<50} {w:>8}")
else:
    print("Set REF_PATH to compute Word Error Rate (WER) against ground-truth text.")

## 8. Save Benchmark results to JSON and individual txt files

In [ ]:
report = {
    "audio": str(WAV_PATH),
    "duration_s": round(duration, 1),
    "results": {
        r["model"]: {
            "load_s": r["load_s"],
            "transcribe_s": r["transcribe_s"],
            "text": r["text"],
            "error": r.get("error"),
        }
        for r in [result_1, result_2, result_baseline]
    },
}
out_path = OUT_DIR / "small_models_benchmark_results.json"
out_path.write_text(json.dumps(report, ensure_ascii=False, indent=2))
print(f"Saved JSON results to {out_path}")

# Save each model's raw text transcript to its own file for individual downloading
for r in [result_1, result_2, result_baseline]:
    tag = r["model"].split("/")[-1]
    txt_path = OUT_DIR / f"{tag}_transcript.txt"
    txt_path.write_text(r.get("text", ""))
    print(f"Saved transcript output to {txt_path}")